# Context Compression

**Prerequisites:** `02_context_selection.ipynb`. From `03_tools`: `03_invocation.ipynb` (for `truncate`)

**Compression makes information smaller. Selection chooses between items; compression changes them.**

Notebook 2 ended on three things selection couldn't do:

- The pinned rules alone overflowed a small budget, and `select()` refused. It can leave things out, but it can't make what must go in any smaller.
- The raw logs, 1,142 tokens, could never fit a 400-token budget, so they were dropped whole, along with everything in them.
- The baseline, *"up from 0.3 s yesterday"*, lived in a reading that was superseded. Dropping the item dropped the fact, and the model invented a baseline twice.

All three need information made smaller without being thrown away. That's compression.

**Compression isn't a stage.** It isn't a box every context passes through on its way to the model. It's an operation that runs when something calls for it:

```
context
   ├── something must go in and can't fit  ──→ compress that thing
   ├── the run has grown past a threshold  ──→ compress the old part
   └── a user or the system asks           ──→ compress
```

Most calls need none of it, and running it anyway costs a model call and loses information for nothing.

**Every method loses something.** The question is never whether information is lost. It's *what* is lost, whether you chose that, and whether the result admits it's incomplete.

## Setup

In [1]:
import sys
from pathlib import Path

HERE = Path.cwd().parent
sys.path.append(str(HERE))
for chapter in ("01_foundations", "02_agent_runtime", "04_memory"):
    sys.path.append(str(HERE.parent / chapter))

from compression import compact, extract_facts, missing, salvage, summarize, truncate, why_compress
from context import available, render, tokens
from incident import FACTS, RULES, incident
from llm import DEFAULT_MODEL, client
from selection import select, supersede

conversation, pad = incident()
items = available(conversation, pad, constraints=RULES)


def ask(items):
    # the same stand-in for construction as notebooks 1 and 2
    system = "\n".join(i.content for i in items if i.kind == "instruction")
    rest = [i for i in items if i.kind != "instruction"]
    response = client.chat.completions.create(model=DEFAULT_MODEL, messages=[
        {"role": "system", "content": system},
        {"role": "user", "content": render(rest)},
    ])
    print(f"prompt {response.usage.prompt_tokens} tokens\n")
    print(response.choices[0].message.content)
    return response

## Measuring what was lost

Reading a summary and deciding it "looks fine" is how information loss goes unnoticed: a summary reads fine *because* you can't see what it left out. So before compressing anything, the notebook needs a way to measure it.

`incident.FACTS` lists what a context about this incident should still be able to say, each with a pattern that finds it in text. `missing(items, FACTS)` names the ones that no longer appear anywhere. It's a floor, not a judge: it can tell that a number survived, not that it's still attached to the right thing. But it turns "did we lose anything?" into a list.

In [2]:
for name, pattern in FACTS.items():
    print(f"{name:<28} {pattern}")

recent = select(items, budget=250, w_relevance=0, w_importance=0, w_recency=1)
print(f"\neverything             {tokens(items):>5} tokens   missing: {missing(items, FACTS)}")
print(f"notebook 2's recency   {recent.tokens:>5} tokens   missing: {missing(recent.kept, FACTS)}")

cause: deploy v4.12.0        v4\.12\.0
pool cut from 50 to 5        \b50\b[^.\n]{0,20}\b5\b
baseline: 0.3 s yesterday    0\.3\s*s|\b300\s*ms
p95 now 2.9 s                2\.9\s*s|2[,\s]?900\s*ms
errors began 07:12           07:12
error rate 3.4%              3\.4\s*%|0\.034
orders-db is healthy         orders-db[^\n]{0,80}healthy
rule: milliseconds           millisecond
rule: read-only              read-only|read only

everything              2219 tokens   missing: []
notebook 2's recency     237 tokens   missing: ['baseline: 0.3 s yesterday', 'errors began 07:12', 'error rate 3.4%']


The full session has all nine. Notebook 2's recency selection, with the rules pinned, has six: it lost the baseline, the time the errors began, and the error rate. The cause survives only because the scratchpad's `suspect` entry repeats it. Every compression below is measured the same way.

## When to compress

`why_compress()` reports which triggers apply. It never compresses anything itself, because the decision to compress is the caller's, and a user asking for it is a reason no function can see.

In [3]:
for reason in why_compress(items, budget=400, history_limit=1500):
    print("-", reason)
print()
for reason in why_compress(items, budget=50):
    print("-", reason)

- t6 tool_result is 1142 tokens and can never fit 400
- the run holds 2032 tokens, past the limit of 1500

- pinned items are 65 tokens, over the budget of 50
- t6 tool_result is 1142 tokens and can never fit 50
- t8 tool_result is 119 tokens and can never fit 50
- t9 message is 52 tokens and can never fit 50
- t12 tool_result is 143 tokens and can never fit 50
- t17 tool_result is 57 tokens and can never fit 50
- t19 tool_result is 146 tokens and can never fit 50


At a 400-token budget, one item can never fit: the raw logs. And the run as a whole is past a 1,500-token history limit. At 50 tokens the pinned items themselves don't fit, and neither does any raw result. Each reason points at a different operation, so they're taken one at a time.

## Truncation

**Truncation keeps an exact piece and throws away the rest.** It's free, it's deterministic, and it never changes a word it keeps. Chapter 3's `invoke.truncate()` keeps the head of a tool result and appends a note saying how much was cut, because without the note a model reads a cut result as a complete one.

A log is read the other way round from most tool results: the newest lines are at the end. So `truncate()` here takes `keep="head"`, `"tail"` or `"ends"`. The logs, cut to 150 tokens each way, and measured against what they say:

In [4]:
logs = next(i for i in items if i.kind == "tool_result" and i.about.startswith("get_logs"))

LOG_FACTS = {
    "errors began 07:12": r"07:12",
    "latest: 32 waiting": r"\b32\b",
    "pool exhausted": r"pool exhausted",
    "retries after timeouts": r"retr(y|ied|ies|ying)",
    "requests take ~2 s": r"\b2,?[0-9]{3}\s?ms|\b2\.[0-9]+\s?s\b",
}

for keep in ("head", "tail", "ends"):
    cut = truncate(logs, 150, keep=keep)
    print(f"--- keep={keep}: {cut.tokens} tokens, missing {missing([cut], LOG_FACTS)}")
    lines = cut.content.splitlines()
    print("\n".join(lines[:2] + ["    ..."] + lines[-2:]) if len(lines) > 4 else cut.content)
    print()

--- keep=head: 161 tokens, missing ['latest: 32 waiting', 'retries after timeouts']
07:12:24 INFO  checkout-api  POST /checkout 200 2052ms
07:12:24 ERROR checkout-api  orders-db client: connection pool exhausted (5/5 in use, 3 waiting)
    ...
07:15:45 ERROR checkout-api  orders-db client: connection pool exhausted (5/5 in use, 6 waiting)
[... 994 of 1,142 tokens cut ...]

--- keep=tail: 138 tokens, missing ['errors began 07:12']
[... 1,019 of 1,142 tokens cut ...]
07:39:33 INFO  checkout-api  POST /checkout 200 2619ms
    ...
07:40:40 WARN  checkout-api  retrying order lookup after pool timeout (attempt 2)
07:41:47 ERROR checkout-api  orders-db client: connection pool exhausted (5/5 in use, 32 waiting)

--- keep=ends: 138 tokens, missing []
07:12:24 INFO  checkout-api  POST /checkout 200 2052ms
07:12:24 ERROR checkout-api  orders-db client: connection pool exhausted (5/5 in use, 3 waiting)
    ...
07:40:40 WARN  checkout-api  retrying order lookup after pool timeout (attempt 2)
07:41:

Each cut is exact, and each keeps a different story. The **head** keeps the onset (07:12, 3 waiting) and loses how bad it got. The **tail** keeps the latest state, 32 waiting, and loses when it began. **Both ends** hold all five facts in 138 tokens, an eighth of the original, because this log is a steady trend and its two ends define it.

A log with one important line in the middle would lose it to all three, and the `[... cut ...]` note would be the only sign. Truncation is the right tool when you know *where* in the content the value is. For a log that's usually the end; for a document, the start.

## Summarization

**A summary keeps the gist of all of it, and decides for itself what the gist is.** It costs a model call, and what it drops is invisible in the result. Here are the logs summarized, first with no purpose given:

In [5]:
plain = summarize([logs])
print(f"{plain.tokens} tokens, missing {missing([plain], LOG_FACTS)}\n")
print(plain.content)

215 tokens, missing []

Asked: Provide a concise summary of the log. Found: From 07:12:24 to 07:41:47 the checkout‑api repeatedly returned POST /checkout 200 with response times 2052ms, 2115ms, 2178ms, 2241ms, 2304ms, 2367ms, 2430ms, 2493ms, 2556ms, 2619ms. Each request was followed by orders‑db client errors: “connection pool exhausted (5/5 in use, X waiting)” where X increased from 3 to 32, and WARN lines retrying order lookup after pool timeout (attempt 2) at 07:15:45, 07:20:20, 07:25:55, 07:30:30, 07:35:05, 07:40:40. Concluded: The database connection pool of size 5 is saturated, causing growing wait queues and retries; increase pool size or fix connection leaks to restore stability.


And with the purpose stated: `focus` says what the summary is for.

In [6]:
focused = summarize([logs], focus="when the errors started, whether they are getting worse, "
                                  "and what exactly is running out")
print(f"{focused.tokens} tokens, missing {missing([focused], LOG_FACTS)}\n")
print(focused.content)

92 tokens, missing ['retries after timeouts', 'requests take ~2 s']

Asked to summarize when errors started, if they worsen, and what runs out. Found errors began at 07:12:24 with “orders-db client: connection pool exhausted (5/5 in use, 3 waiting)”. The waiting count rose each error, reaching 32 waiting by 07:41:47, showing a clear worsening trend. The exhausted resource is the orders‑db client connection pool, limited to 5 simultaneous connections.


With no purpose given, the summary spent most of its 215 tokens listing every request time, one by one. All five log facts survive, but it's barely a summary: a fifth of the size, and mostly copied.

With a purpose it's 92 tokens and answers exactly what was asked: started at 07:12:24, worsening to 32 waiting, a pool of 5 exhausted. It dropped the retries and the request latencies, which the focus didn't ask about.

Neither summary is wrong. Each is right for one reader, and that's the risk in keeping a summary: it was written for the purpose at hand, and the next operation may need what it left out. The request latencies are exactly what "how bad is it for users" would need. And neither summary says what it left out. Both read as complete; only measuring against a list showed the difference.

## Fact extraction

**Extraction keeps individual statements and drops everything that connects them.** Not a shorter version of the text, but a list of what it asserted, one sentence each. That makes it the right tool for two jobs from notebook 2.

**The first job is noticing a rule.** In notebook 2 the rules only became constraints because the notebook's author passed `constraints=RULES`. In a live system something has to notice a rule when it's said. Here is the t1 message on its own:

In [5]:
first = next(i for i in items if i.turn == 1)
print(first.content, "\n")

noticed = extract_facts([first])
for f in noticed:
    print(f"[{f.kind}{', pinned' if f.pinned else ''}] {f.content}")

print()
for rule in RULES:
    print(f"verbatim? {any(f.content.rstrip('.') == rule.rstrip('.') for f in noticed if f.kind == 'constraint')}  {rule}")

user: Checkout has been slow since this morning. Can you look into it? Two rules for this whole session: Everything you do must be read-only: do not restart, scale or redeploy anything in production. Give every duration in milliseconds. 



[constraint, pinned] Everything you do must be read-only: do not restart, scale or redeploy anything in production.
[constraint, pinned] Give every duration in milliseconds.
[fact] Checkout has been slow since this morning.

verbatim? True  Everything you do must be read-only: do not restart, scale or redeploy anything in production.
verbatim? True  Give every duration in milliseconds.


Both rules come back as constraints, word for word, and pinned. That's the extraction notebook 2 promised: the rules no longer depend on someone passing `constraints=RULES`.

*Word for word* is doing real work: it makes the extraction **checkable**. A constraint that doesn't appear verbatim in its source was either paraphrased or invented, and `extract_facts()` downgrades it to an ordinary, unpinned fact. That check is in code because the prompt alone didn't hold. In one run while this notebook was being written, a line of the agent's own plan, *"the database has to be ruled out before blaming the deploy"*, came back as a pinned constraint, despite a prompt saying in capitals that plans never are. Pinned means never dropped, so a wrong pin is a permanent mistake.

**The second job is saving facts from items that are about to be dropped.** Superseding works on whole items. The t4 reading was replaced because its *value* was out of date, but the same sentence carried a baseline that wasn't, and nothing else said it. The obvious move is to extract from the items before supersession drops them:

In [6]:
current, old = supersede(items)
dropped = [item for item, _ in old]
for item in dropped:
    print(f"dropping t{item.turn} {item.kind}: {' '.join(item.content.split())[:70]}")

naive = extract_facts(dropped)
print()
for f in naive:
    print(f"[{f.kind}] {f.content}")

dropping t2 state: Goals: 0. Check checkout-api latency 1. Read checkout-api logs 2. Iden
dropping t4 observation: checkout-api p95 latency is 2.4 s, up from 0.3 s yesterday.
dropping t4 tool_result: { "service": "checkout-api", "window": "last 15m", "latency_ms": { "p5



[fact] checkout-api p95 latency is 2.4 s, up from 0.3 s yesterday.
[fact] checkout-api p50 latency in the last 15m is 910 ms.
[fact] checkout-api p95 latency in the last 15m is 2400 ms.
[fact] checkout-api p99 latency in the last 15m is 3840 ms.
[fact] checkout-api handled 140 requests per second in the last 15m.
[fact] checkout-api error rate in the last 15m is 0.021.
[fact] checkout-api has 6 instances in the last 15m.
[fact] checkout-api CPU usage across its 6 instances is 31, 29, 34, 30, 33, and 28 percent.
[fact] checkout-api memory usage across its 6 instances is 612, 598, 640, 605, 611, and 620 MB.


The baseline is there, on the first line. So is everything supersession removed, back as present-tense fact: p95 "in the last 15m is 2400 ms", an error rate of 0.021, 140 requests a second. Those are the *replaced* values. Put this list in a context beside the current reading, and the model has two answers to "how slow is it" again, with nothing marking the old one. In an earlier run of this notebook's compaction, the final answer quoted the stale p99 and request rate as current.

Extraction can't tell what's out of date, because it can't see what replaced it. So show it:

In [7]:
versions = {(i.kind, i.about) for i in dropped}
newer = [i for i in current if (i.kind, i.about) in versions]

salvaged = salvage(dropped, newer)
for f in salvaged:
    print(f"[{f.kind}] {f.content}")

print(f"\nbaseline after superseding alone      : "
      f"{'kept' if 'baseline: 0.3 s yesterday' not in missing(current, FACTS) else 'lost'}")
print(f"baseline after superseding + salvage  : "
      f"{'kept' if 'baseline: 0.3 s yesterday' not in missing(current + salvaged, FACTS) else 'lost'}")

[fact] checkout-api p95 latency was 0.3 s yesterday.

baseline after superseding alone      : lost
baseline after superseding + salvage  : kept


One fact, the baseline, in the past tense. `salvage()` puts each old item beside the newer version of the same thing and keeps only what the newer one neither restates nor updates. The old raw metrics are replaced field for field, so nothing survives from them; the old plan is replaced whole. What's left is exactly the one thing notebook 2 lost.

## Preserving constraints and task state

Two kinds of item are never handed to a summarizer, and the reasons differ:

- **Constraints**: a summary is a paraphrase, and a paraphrased rule is a different rule. *"Don't change production"* and *"do not restart, scale or redeploy anything in production"* permit different things. A constraint is kept word for word or not at all. When the pinned rules overflow a budget, as at 50 tokens above, the answer is a bigger budget or fewer rules. There's no safe way to shrink one.
- **Task state**: the current plan and the scratchpad are *already* compressed. They're the run's own summary of where things stand, written by the agent as it went, and the next step starts from them. Summarizing a plan loses its step numbers, and "carry on from step 2" stops meaning anything.

Neither of these is a model's judgement call. They're rules about what compression is allowed to touch, and `compact()` enforces them in code.

## Conversation compression

**`compact()` compresses everything older than the last few turns and leaves the rest alone.** Each kind of item goes where it's safest:

| Items | What happens | Why |
|---|---|---|
| pinned (rules, instruction, request) | untouched | a paraphrased rule is a different rule |
| task state (plan, scratchpad) | untouched | already the run's own summary |
| the last `keep_recent` turns | untouched | what the next step builds on, verbatim |
| older messages and observations | summarized | they carry the story |
| older raw tool results | facts extracted | a summary would smooth their numbers away |
| superseded items | salvaged, beside their replacements | so a replaced value doesn't take its neighbours with it, or come back |

Three model calls: one summary, one extraction, one salvage.

In [7]:
compacted = compact(items, keep_recent=6)
print(f"{tokens(items)} tokens -> {compacted.tokens}\n")
print(compacted.report())

2219 tokens -> 858

1813 tokens in 14 items replaced by 452 tokens: a summary and 10 facts

[summary, summary of t1-t14]
User asked why checkout is slow and to investigate read‑only. Observed checkout‑api logs from 07:00 show repeated “connection pool exhausted” errors from the orders‑db client starting at 07:12. The last checkout‑api deploy was v4.12.0 at 07:05, which changed the orders‑db pool size from 50 to 5. Checkout‑api p95 latency rose to 2400 ms after the deploy. User asked about search‑api; observed search‑api p95 latency is 180 ms (normal). User then asked to check orders‑db.

[fact] checkout-api deploy version v4.12.0 occurred at 07:05 and changed orders-db pool size from 50 to 5.
[fact] checkout-api p95 latency rose to 2.4 s after deploy v4.12.0 at 07:05.
[fact] checkout-api connection pool exhausted errors from the orders-db client started at 07:12 and increased waiting count from 3 to 32, persisting until 07:41:47.
[fact] checkout-api POST /checkout latency increased fro

The things that must not change are checked in code, not by reading:

In [8]:
untouched = [i for i in items if i.pinned or (i.kind == "state" and i.turn >= 15) or i.turn > 16]
assert all(any(kept is i for kept in compacted.items) for i in untouched)
print(f"{len(untouched)} pinned, state and recent items passed through as the same objects")
print(f"missing after compaction: {missing(compacted.items, FACTS)}")

12 pinned, state and recent items passed through as the same objects
missing after compaction: ['baseline: 0.3 s yesterday']


858 tokens, down from 2,219: the story in one summary and ten facts. The log became a single fact, with its start, trend and end, instead of one fact per line. The prompt asks for that because the first run of `compact()` produced 28 facts, 17 of them a single log line each, and only halved the session. Everything that must not change passed through as the same objects.

**One fact is missing: the baseline.** Salvage ran inside `compact()`, and this time it returned three facts about the old reading, correctly labelled "during the old observation", and not the baseline. The standalone salvage above found it; so did earlier runs of this same compaction. Across the runs made while writing this notebook it caught the baseline three times out of four. That's what a model-based compression step is: **a sample, not a function.** `missing()` caught it here only because the notebook knows what to look for, which a live system usually doesn't.

What limits the damage is notebook 1's distinction. The conversation is the record, and compaction only changed *this list of items*. The t4 reading is still in `conversation.messages`, untouched. Compression shrinks a context; it must never be allowed to edit the record, because the record is where a lost fact can still be recovered from.

## Information loss, measured

Everything side by side. The last row is what a real call would get: the compacted run, then selected to the same 400-token budget as notebook 2.

In [9]:
selected_after = select(compacted.items, budget=400)
rows = [
    ("everything", items),
    ("notebook 2: recency, 250", recent.kept),
    ("notebook 2: scored, 400", select(items, budget=400).kept),
    ("compacted", compacted.items),
    ("compacted, then scored, 400", selected_after.kept),
]
for name, row in rows:
    gone = missing(row, FACTS)
    print(f"{name:<29}{tokens(row):>5} tokens  {len(FACTS) - len(gone)}/{len(FACTS)} facts"
          + (f"  missing: {', '.join(gone)}" if gone else ""))

everything                    2219 tokens  9/9 facts
notebook 2: recency, 250       237 tokens  6/9 facts  missing: baseline: 0.3 s yesterday, errors began 07:12, error rate 3.4%
notebook 2: scored, 400        390 tokens  8/9 facts  missing: baseline: 0.3 s yesterday
compacted                      858 tokens  8/9 facts  missing: baseline: 0.3 s yesterday
compacted, then scored, 400    399 tokens  6/9 facts  missing: baseline: 0.3 s yesterday, error rate 3.4%, orders-db is healthy


Compaction alone keeps 8 of 9 facts in 858 tokens, 39% of the session. Selecting from that down to 400 tokens keeps only **6 of 9**, which is *worse* than notebook 2's selection from the uncompressed session at the same budget (8 of 9). The ten new facts are small, rank well (importance 0.7), and crowd out the current raw metrics, which carried the 3.4% error rate, and the orders-db observation.

Compression and selection aren't independent. Every item compression creates is one more competitor for the budget, and a relevance score can't see that a fact is the only copy of something. Compressing and then selecting hard can leave less than selecting alone.

## The same question, after compression

Notebook 2's final answer borrowed search-api's 180 ms as checkout's baseline, because the real one had been superseded away. Here is the question again, on the compacted run selected to 400 tokens:

In [10]:
answer = ask(selected_after.kept)

prompt 503 tokens

The checkout slowdown is caused by the **orders‑db connection pool being exhausted** after the 07:05 v4.12.0 deploy reduced the pool size from 50 to 5. From 07:12 until 07:41:47 the “connection pool exhausted” errors rose, with the waiting‑request count climbing from 3 to 32, forcing each checkout request to queue for additional DB time. That queuing pushed the **checkout‑api p95 latency to 2 900 ms** (now) and the POST /checkout latency from 2 052 ms at 07:12:24 to 2 619 ms at 07:39:33.

**Recommendation (read‑only):** Restore the orders‑db pool size to its original value (≈ 50) or increase it enough to handle the observed concurrent load, then monitor the pool‑exhausted error rate and latency. If a larger pool isn’t feasible, consider rate‑limiting checkout traffic or optimizing DB queries until the pool can be safely expanded. This should immediately eliminate the queueing delay and bring p95 latency back toward the pre‑deploy level (≈ 2 000 ms or lower).


Right cause, right evidence, milliseconds throughout. And another made-up baseline: *"back toward the pre-deploy level (≈ 2 000 ms or lower)"*. The real one is 300 ms. Neither this run's compaction nor the selection kept it, and asked how bad it is, the model filled the gap: the third time in this chapter.

The fairer test of compression by itself: the whole compacted run, 858 tokens, with no selection on top.

In [11]:
whole = ask(compacted.items)

prompt 997 tokens

The checkout slowdown is being driven by the **v4.12.0 deployment** that reduced the orders‑db client pool from **50 connections to 5**. Starting at 07:12 the service began logging “connection pool exhausted” errors, the waiting‑connection count grew from 3 to 32, and latency for the POST /checkout endpoint climbed from 2052 ms to 2619 ms and now sits near the service‑wide p95 of **2900 ms** (p50 ≈ 1150 ms, p99 ≈ 4640 ms) with an error rate of **3.4 %**.

**How bad it is now**  
* checkout‑api p95 latency ≈ 2900 ms (≈ 2.9 s)  
* error rate ≈ 0.034 (3.4 %)  
* request rate ≈ 138 RPS across 6 instances (CPU ≈ 30 % average, memory ≈ 615 MB per instance)  

**What to do (read‑only recommendation)**  
1. **Increase the orders‑db connection pool back to its original size (≈ 50)** or to a value that matches the observed concurrent usage (≈ 40–50).  
2. If the config change cannot be applied immediately, **roll back or hot‑patch v4.12.0** to restore the larger pool.  
3. Aft

Apart from "everything" in notebook 1, this is the most complete answer in the chapter: the cause, the log trend, the current p50, p95 and p99, the 3.4% error rate, the request rate, and the database cleared. It used 997 prompt tokens, against 2,347 for the whole session.

One thing is wrong, and it's the one thing this compaction lost: *"they should drop back toward pre-deploy levels (p95 ≈ 2.4 s → < 2 s)"*. 2.4 s is the superseded reading; pre-deploy was 300 ms.

For this session, compression kept almost everything in a little over a third of the tokens, and the one fact it dropped is the one the answer got wrong. **Compression works, and what it loses is exactly what the model will get wrong, silently, unless you measure.**

## Triggered, not a stage

After compaction, the triggers that fired at the start of the notebook:

In [12]:
print("before:", why_compress(items, budget=400, history_limit=1500))
print("after :", why_compress(compacted.items, budget=400, history_limit=1500))

before: ['t6 tool_result is 1142 tokens and can never fit 400', 'the run holds 2032 tokens, past the limit of 1500']
after : []


Nothing left to do, so nothing runs. That's what "an operation, not a stage" means in practice: compression happens when a trigger fires or someone asks, and costs nothing the rest of the time. Wiring it in as a step every call passes through would pay a model call per turn to re-summarize a run that fits.

## What you built

`compression.py`:

- **`truncate(item, max_tokens, keep=)`**: head, tail or both ends, cut at line breaks, with a note saying what was cut. Free and exact. Which end to keep is a fact about the content.
- **`summarize(items, focus=)`**: one `summary` item. `focus` is what the summary is *for*, and what it keeps depends on it.
- **`extract_facts(items, known=)`**: one sentence per fact, and constraints word for word and pinned. It noticed the rules in t1 that notebook 2 had to pass in by hand, and it named the rules word for word.
- **`salvage(old, newer)`**: extraction from superseded items with their replacements in view, keeping only what the new version doesn't restate. Extracting from them blind brought the stale values back.
- **`compact(items, keep_recent=)`**: conversation compression that never touches pinned items, task state or recent turns, summarizes the story, extracts facts from old raw results, and salvages superseded items.
- **`why_compress(items, budget, history_limit)`**: reports the triggers. It never acts on them.
- **`missing(items, checks)`**: information loss as a list, instead of a feeling about a summary.

Everything so far has produced *items*: selected, retrieved, compressed. Nothing has decided how they're laid out for the model. They still arrive as one user message of labelled lines, with rules, data and a tool's output all mixed together. Notebook 4 is construction: ordering, formatting, and keeping instructions apart from data.